# Four-market Heston sampling calibration

Review the workstation sweep, explicitly choose one interval for each market, then inspect the final independent-ensemble results. This notebook reads CSVs and PNGs; optimization and simulations run on the workstation.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
DATA = ROOT / 'data/processed/heston_calibration/four_market_sampling_sweep'
FIGURES = ROOT / 'visualization/heston_calibration/four_market_sampling_sweep'
MARKETS = ('UN', 'UW', 'LN', 'JT')
summary = {}
for market in MARKETS:
    path = DATA / market / f'{market}_sampling_fit_sweep.csv'
    if path.is_file():
        frame = pd.read_csv(path).sort_values('interval_steps').reset_index(drop=True)
        frame['interval_steps'] = frame.interval_steps.astype(int)
        frame['validation_loss'] = pd.to_numeric(frame.validation_loss, errors='coerce')
        summary[market] = frame
    else:
        print(f'Missing sweep summary: {path}')
print('Available markets:', ', '.join(summary) or 'none')


## Compare fit scores and return scale

`validation_loss` is measured on an independent full-length synthetic ensemble. Compare it with MFHT coverage and the return PDF before choosing an interval.


In [ ]:
for market, frame in summary.items():
    print(market)
    display(frame[['interval_steps', 'pilot_loss', 'validation_loss', 'optimization_success',
                   'empirical_average_std', 'synthetic_average_std',
                   'empirical_mass_abs_le_0p0025', 'synthetic_mass_abs_le_0p0025',
                   'a', 'b', 'aa', 'bb', 'cc', 'vstart']].round(5))
if summary:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), layout='constrained')
    for ax, market in zip(axes.flat, MARKETS):
        if market not in summary:
            ax.set_visible(False)
            continue
        frame = summary[market]
        ax.plot(frame.interval_steps, frame.pilot_loss, 'o-', label='Pilot')
        ax.plot(frame.interval_steps, frame.validation_loss, 's-', label='Validation')
        ax.set(title=market, xlabel='Euler steps per observed day', ylabel='Combined loss',
               xticks=frame.interval_steps)
        ax.legend()
    plt.show()


## Inspect MFHT curves and return PDFs

Each row is one market; columns are the fitted sampling intervals. MFHT titles show covered empirical fit bins.


In [ ]:
for market in MARKETS:
    if market not in summary:
        continue
    intervals = summary[market].interval_steps.tolist()
    fig, axes = plt.subplots(2, len(intervals), figsize=(4 * len(intervals), 7), layout='constrained')
    axes = np.asarray(axes).reshape(2, len(intervals))
    for column, interval in enumerate(intervals):
        curve_path = FIGURES / market / f'interval_{interval}' / market / f'{market}_m0p1_m1p5_mfht.csv'
        pdf_path = FIGURES / market / f'interval_{interval}' / market / f'{market}_returns_pdf.png'
        if curve_path.is_file():
            curve = pd.read_csv(curve_path)
            fit = curve.fit_bin.astype(bool)
            covered = curve.simulated_bin_covered.astype(bool)
            axes[0, column].plot(curve.loc[fit, 'volatility_midpoint'],
                                 curve.loc[fit, 'empirical_mfht'], label='Empirical')
            axes[0, column].plot(curve.volatility_midpoint, curve.simulated_mfht, label='Synthetic')
            axes[0, column].set(title=f'{market} {interval}dt: {(fit & covered).sum()}/{fit.sum()} bins',
                                xlabel='Local volatility', ylabel='MFHT')
            axes[0, column].legend()
        else:
            axes[0, column].set(title=f'Missing {curve_path.name}')
        if pdf_path.is_file():
            axes[1, column].imshow(plt.imread(pdf_path))
        else:
            axes[1, column].set(title=f'Missing {pdf_path.name}')
        axes[1, column].axis('off')
    plt.show()


## Choose one interval per market

Edit the four values below after reviewing the scores and plots. Running the cell writes `final_selection.json`; copy that file to the workstation's matching sweep folder.


In [ ]:
SELECTED_INTERVALS = {'UN': None, 'UW': None, 'LN': None, 'JT': None}

if any(value is None for value in SELECTED_INTERVALS.values()):
    print('Set all four SELECTED_INTERVALS values to 1, 2, 5, or 10, then rerun this cell.')
else:
    if set(SELECTED_INTERVALS) != set(MARKETS):
        raise ValueError('Selection must contain exactly UN, UW, LN, and JT')
    selected_rows = []
    for market, interval in SELECTED_INTERVALS.items():
        if type(interval) is not int or market not in summary:
            raise ValueError(f'Invalid or unavailable market selection: {market}={interval}')
        matches = summary[market].loc[summary[market].interval_steps == interval]
        if len(matches) != 1 or not np.isfinite(matches.validation_loss.iloc[0]):
            raise ValueError(f'{market} {interval}dt has no finite full validation score')
        parameters = DATA / market / f'interval_{interval}' / 'heston_calibration_parameters.csv'
        if not parameters.is_file():
            raise FileNotFoundError(parameters)
        selected_rows.append({'market': market, 'interval_steps': interval,
                              'validation_loss': matches.validation_loss.iloc[0],
                              'parameters_csv': str(parameters)})
    display(pd.DataFrame(selected_rows))
    selection_path = DATA / 'final_selection.json'
    selection_path.write_text(json.dumps(SELECTED_INTERVALS, indent=2) + '\n', encoding='utf-8')
    print('Saved:', selection_path)


## Final independent-ensemble comparison

The workstation script produces 10 full-length ensembles for each selected model. Shading is ± one sample standard deviation across runs. Scores are reported as across-run mean and standard deviation.


In [ ]:
FINAL = DATA / 'replicate_variation'
score_path = FINAL / 'replicate_summary.csv'
selection_path = DATA / 'final_selection.json'
if not score_path.is_file() or not selection_path.is_file():
    print('Copy the workstation replicate_variation folder and final_selection.json here first.')
else:
    chosen = json.loads(selection_path.read_text(encoding='utf-8'))
    scores = pd.read_csv(score_path).set_index('market')
    display(scores[['interval_steps', 'return_pdf_total_variation_mean',
                    'return_pdf_total_variation_std', 'return_wasserstein_sampled_mean',
                    'return_wasserstein_sampled_std', 'return_loss_mean', 'return_loss_std',
                    'mfht_loss_mean', 'mfht_loss_std', 'combined_loss_mean',
                    'combined_loss_std', 'n_events_mean']].round(5))
    fig, axes = plt.subplots(4, 2, figsize=(13, 15), layout='constrained')
    for row, market in enumerate(MARKETS):
        interval = chosen[market]
        folder = FINAL / market / f'interval_{interval}'
        for column, (name, x, empirical, xlabel, ylabel) in enumerate([
            ('return_pdf_band.csv', 'return_midpoint', 'empirical_density', 'Daily return', 'Density'),
            ('mfht_band.csv', 'volatility_midpoint', 'empirical_mfht', 'Local volatility', 'MFHT')]):
            frame = pd.read_csv(folder / name)
            ax = axes[row, column]
            xx = frame[x].to_numpy(dtype=float)
            yy = frame.synthetic_mean.to_numpy(dtype=float)
            sd = frame.synthetic_std.to_numpy(dtype=float)
            ax.plot(xx, frame[empirical], label='Empirical')
            ax.plot(xx, yy, label='Synthetic mean')
            ax.fill_between(xx, yy - sd, yy + sd, alpha=0.25, label='±1 SD')
            ax.set(title=f'{market}, {interval}dt', xlabel=xlabel, ylabel=ylabel)
            if column == 0:
                ax.set_xlim(-0.05, 0.05)
            ax.legend()
    plt.show()


## Optional return-PDF rebinning

This cell reads one saved full return matrix only when you choose a market. It is more memory intensive than the summary plots above.


In [ ]:
REBIN_MARKET = None  # e.g. 'UN'
BINS = 240
if REBIN_MARKET is not None:
    chosen = json.loads((DATA / 'final_selection.json').read_text(encoding='utf-8'))
    interval = chosen[REBIN_MARKET]
    synthetic_path = FINAL / REBIN_MARKET / f'interval_{interval}' / f'{REBIN_MARKET}_returns_replicate_001.pkl'
    empirical = pd.read_pickle(ROOT / 'data/interim' / f'{REBIN_MARKET}.pickle').loc['1980-01-01':'2022-07-01']
    empirical = empirical.loc[:, empirical.notna().sum() >= 3030]
    real = empirical.to_numpy(dtype=float, copy=False).ravel()
    real = real[np.isfinite(real)]
    synthetic = pd.read_pickle(synthetic_path).to_numpy(dtype=float, copy=False).ravel()
    synthetic = synthetic[np.isfinite(synthetic)]
    edges = np.linspace(*np.quantile(real, [0.001, 0.999]), BINS + 1)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(real, bins=edges, density=True, histtype='step', label='Empirical')
    ax.hist(synthetic, bins=edges, density=True, histtype='step', label='Synthetic run 1')
    ax.set(title=f'{REBIN_MARKET}: {BINS} bins', xlabel='Daily return', ylabel='Density', xlim=(-0.05, 0.05))
    ax.legend()
    plt.show()
